# 03 · Tool Guardrails

Input guards try to stop attacks from getting in. But some will get through, and sometimes an agent just makes a mistake. So the next layer asks: **what is the worst this agent can do?**

A tool guard limits that, whatever the agent says or is told. Three simple kinds:

| Guard | Example |
|---|---|
| **Limits** | A refund above $100 is not issued automatically |
| **Human approval** | A person approves risky actions |
| **Budgets** | A cap on steps and tokens, so a stuck agent cannot run up the bill |

In [1]:
import shop
import guards
from shop import client, MODEL

print("Ready")

Ready


In [2]:
from shop import orders_agent, call_tool, make_tool, run_agent
from desk import handle_email

## A refund limit with human approval
In notebook 01 the agent refunded a $250 television on its own. The guarded version checks the amount. Above `MAX_AUTO_REFUND` it does **not** refund. It puts the order in a waiting list for a human.

Notice that `approve_refund` is a **separate function that agents never receive**. Only a person (or the back-office code) can call it.

In [3]:
from shop import ORDERS, refund_check, issue_refund
from guards import MAX_AUTO_REFUND, PENDING_APPROVALS      # the limit, and the list of orders waiting for a human

In [4]:
def issue_refund_guarded(order_id):
    """Like issue_refund, but a refund above the limit waits for a human to approve it."""
    order_id = order_id.strip().upper()

    allowed, explanation = refund_check(order_id)
    if not allowed:
        return "Refund refused. " + explanation

    order = ORDERS[order_id]
    if order["price"] > MAX_AUTO_REFUND:
        if order_id not in PENDING_APPROVALS:
            PENDING_APPROVALS.append(order_id)
        return (
            "This refund is above the limit of $" + format(MAX_AUTO_REFUND, ".2f")
            + ". It has been sent to a human for approval and has NOT been issued yet."
        )

    return issue_refund(order_id)


def approve_refund(order_id):
    """A HUMAN approves a waiting refund. Agents are never given this function."""
    order_id = order_id.strip().upper()
    if order_id not in PENDING_APPROVALS:
        return "Nothing is waiting for approval for order " + order_id + "."

    PENDING_APPROVALS.remove(order_id)
    return issue_refund(order_id)

In [5]:
shop.reset_shop()
guards.PENDING_APPROVALS.clear()

print("A1003 ($45, 80 days old):", issue_refund_guarded("A1003"))
print("A1001 ($79, small):      ", issue_refund_guarded("A1001"))
print("A1004 ($250, large):     ", issue_refund_guarded("A1004"))

print()
print("Waiting for a human:", PENDING_APPROVALS)
print("Issued so far:      ", shop.REFUNDS)

A1003 ($45, 80 days old): Refund refused. Not eligible: delivered 80 days ago, the refund window is 30 days.
A1001 ($79, small):       Refund of $79.00 issued for order A1001.
A1004 ($250, large):      This refund is above the limit of $100.00. It has been sent to a human for approval and has NOT been issued yet.

Waiting for a human: ['A1004']
Issued so far:       [{'order_id': 'A1001', 'amount': 79.0}]


Now a person reviews the waiting refund and approves it:

In [6]:
print(approve_refund("A1004"))
print("Issued now:", shop.REFUNDS)

Refund of $250.00 issued for order A1004.
Issued now: [{'order_id': 'A1001', 'amount': 79.0}, {'order_id': 'A1004', 'amount': 250.0}]


## The same attack, with the guard in place
The Refund agent is the same agent with the same instructions. Only the function behind `issue_refund` changed. We run the television email through the guarded desk:

In [7]:
shop.reset_shop()
guards.PENDING_APPROVALS.clear()

email = "Please refund my 4K television, order A1004. It arrived a few days ago but I changed my mind."
result = handle_email(email, guarded=True)

print("Refunds issued:      ", result["refunds"])
print("Waiting for a human: ", result["pending"])

Customer: Please refund my 4K television, order A1004. It arrived a few days ago but I changed my mind.


  [Router] -> refund (Customer requests a refund for their order A1004.)


  [Refund agent] lookup_order({'order_id': 'A1004'})


  [Refund agent] check_refund_eligibility({'order_id': 'A1004'})


  [Refund agent] issue_refund({'order_id': 'A1004'})


  [Output guard] round 1: approved
  Reply sent: Your refund request for the 4K television has been sent for approval since it exceeds $100.00. You'll be notified once a decision has been made. Thank you for your patience!

Refunds issued:       []
Waiting for a human:  ['A1004']


The customer is told their request has been sent for approval, and **no money moved**.

## Agents can only use their own tools
Each agent has a list of tools. `call_tool` refuses anything else. Even if an attacker convinces the **Orders agent** to try a refund, there is no refund tool to run:

In [8]:
print(call_tool(orders_agent["functions"], "issue_refund", {"order_id": "A1001"}))

Unknown tool: issue_refund


This is why we gave each agent only the tools it needs in Module 04. It is a guardrail as well as a design choice.

## Budgets: stop a stuck agent
Agents can get stuck. Here is an agent with one tool that **always** says "still being prepared". Its instructions tell it never to give up, so it would check forever, and every check costs tokens.

In [9]:
def check_delivery_status(order_id):
    return "Status: still being prepared. Please check again."

stuck_agent = {
    "name": "Stuck agent",
    "instructions": "Check the delivery status of the customer's order until it says delivered. Never give up.",
    "tools": [make_tool("check_delivery_status", "Check the delivery status of an order.", "order_id", "The order id")],
    "functions": {"check_delivery_status": check_delivery_status},
}

`run_agent` already has two limits built in. `max_steps` caps the number of laps, and `max_tokens` caps the total tokens used. Here is the step limit, with a small limit of 3 so the output stays short:

In [10]:
shop.reset_shop()

answer = shop.run_agent(stuck_agent, "Is order A1002 delivered yet?", max_steps=3)

print("\nAnswer:", answer)
print("Tokens used:", shop.USAGE["tokens"])

  [Stuck agent] check_delivery_status({'order_id': 'A1002'})


  [Stuck agent] check_delivery_status({'order_id': 'A1002'})


  [Stuck agent] check_delivery_status({'order_id': 'A1002'})

Answer: Stopped: too many steps.
Tokens used: 396


It stopped after 3 laps instead of looping forever. The token budget is a second net, in case one lap is very expensive:

In [11]:
shop.reset_shop()

answer = shop.run_agent(stuck_agent, "Is order A1002 delivered yet?", max_steps=10, max_tokens=700)

print("\nAnswer:", answer)
print("Tokens used:", shop.USAGE["tokens"])

  [Stuck agent] check_delivery_status({'order_id': 'A1002'})


  [Stuck agent] check_delivery_status({'order_id': 'A1002'})


  [Stuck agent] check_delivery_status({'order_id': 'A1002'})


  [Stuck agent] check_delivery_status({'order_id': 'A1002'})



Answer: Stopped: the token budget was used up.
Tokens used: 845


## Recap
* Ask: **what is the worst this agent could do?** Then make the worst case small.
* Use **limits** and a **human approval** step for actions with real consequences, such as refunds, deletions and emails sent to customers.
* The approval function belongs to **people**. Agents never get it.
* Give each agent **only its own tools**.
* Always set a **step limit and a token budget**. A stuck agent costs money every lap.
* Make tools **check the rules themselves** (the refund policy check from Module 02). Do not rely on the agent to follow the rules.

Next: **output guards**, which check the reply before the customer sees it.